In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import gzip
import pickle
import numpy as np
from pathlib import Path
from copy import deepcopy
from time import strftime
from functools import partial
from skimage import io as skio
from scipy.ndimage import binary_fill_holes
from skimage.morphology import skeletonize
from tqdm.auto import tqdm

from shutil import copyfile
from gimpformats.gimpXcfDocument import GimpDocument
import pydicom as dicom
import matplotlib.pylab as plt
from matplotlib.colors import ListedColormap

import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from torchvision import models

from utils import color_codes, normalise, load_compressed_pickle, save_compressed_pickle, load_xcf
from datasets import FetalDataset
from models import FCN_ResNet50, FCN_ResNet101

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data consolidation
The following cells focus on creating a unified dataset combining with DICOM files (.dcm), any available segmentations in the DICOM space (.png) and the point annotations in the DICOM space (.pkl.gz and .csv). We are keeping the DICOMs and point files to be able to use pixel spacing. If we are ever going to measure anything, we need to know how big each pixel is. Furtehrmore, I do believe that if we end up making an atlas, we need to be extra careful with spacing when doing registration.

NOTE: There was a prior manual check to ensure that the DICOM files had matching annotations and some minor corrections for point annotations.

0: Background

1: Cavum

2: Cerebellum

3: Cisterna magna

4: Nuchal fold

5: Midline

6: Sylvian fissure

In [2]:
xcf_path = '/Users/ikaros/Documents/UdG/Datasets/USFoetal2026/Fetal_XCF/'
dicom_path = '/Users/ikaros/Documents/UdG/Datasets/USFoetal2026/Fetal_DCM/'
out_path = '/Users/ikaros/Documents/UdG/Datasets/USFoetal2026/USFoetalSorted/'

xcf_files = sorted([f for f in os.listdir(xcf_path) if f.endswith('.xcf')], key=lambda x: int(x.split('.')[0]))
#print(xcf_files)

image_path = os.path.join(out_path, 'Annotations')
Path(image_path).mkdir(parents=True, exist_ok=True)

layer_list = ['cavum', 'cerebel', 'cisterna magna', 'plec nucal', 'midline', 'silvio']
layer_names = ['cavum', 'cerebellum', 'cisterna magna', 'nuchal fold', 'midline', 'sylvian']
color_list = [[0, 0, 0, 0], 'lime', 'orange', 'magenta', 'blue', 'cyan', 'red']
segmentations = 0
subjects = []
global_dict = {}
labels = []

cmap = ListedColormap(color_list, name='foetal_us')

xcf_loop = tqdm(
    enumerate(xcf_files), total=len(list(xcf_files)), leave=False,
    desc='XCF and DCM file reading',
)

for i, xcf_f in xcf_loop:
    subject_dict = {}
    f_split = xcf_f.split('.')
    sub = f_split[0]
    if sub not in subjects:
        subjects.append(sub)
        global_dict[sub] = []
    subj_code = '.'.join(f_split[:-1])
    dicom_f = subj_code + '.dcm'
    dicom_filepath = os.path.join(dicom_path, dicom_f)
    try:
        xcf_filepath = os.path.join(xcf_path, xcf_f)
        names, data = load_xcf(xcf_filepath)
        layer_dict = {
            name.lower(): d_i
            for name, d_i in zip(names, data)
            if '.dcm' not in name
        }
        labels += sorted(list(layer_dict.keys()))
        ds = dicom.dcmread(dicom_filepath)
        dcm_image = np.mean(ds.pixel_array, axis=-1)
        final_seg = np.max([
            (i + 1) * binary_fill_holes(np.array(layer_dict[name])[..., -1] > 0)
            for i, name in enumerate(layer_list[:-2])
        ], axis=0)
        plot_seg = np.max([
            (i + 1) * binary_fill_holes(np.array(layer_dict[name])[..., -1] > 0)
            for i, name in enumerate(layer_list)
        ], axis=0) 
        
        line_seg = [
            np.array(layer_dict[name])[..., -1] > 0
            for name in layer_list[-2:]
        ]

        # if i % 20 == 0:
        # Plotting the segmentations
        fig, axes = plt.subplots(1, 2, figsize=(40, 20))
        plt.subplot(1, 2, 1)
        plt.imshow(dcm_image, cmap='gray')
        plt.imshow(plot_seg, cmap=cmap, vmin=0, alpha=0.50)
        axes[0].set_title('Segmentation (Subject {:>2})'.format(subj_code))
        axes[0].axis("off")
        plt.subplot(1, 2, 2)
        plt.imshow(dcm_image, cmap='gray')
        axes[1].set_title('DICOM (Subject {:>2})'.format(subj_code))
        axes[1].axis("off")
        print('{:<6}'.format(subj_code), sorted(layer_dict.keys()), final_seg.shape, dcm_image.shape)
        plt.tight_layout()
        plt.savefig(os.path.join(image_path, f'{subj_code}_plot.png'))
        plt.close()

        seg_filepath = os.path.join(out_path, subj_code + '.png')
        skio.imsave(seg_filepath, final_seg.astype(np.uint8))
        dicom_outpath = os.path.join(out_path, dicom_f)
        copyfile(dicom_filepath, dicom_outpath)
        global_dict[sub].append((subj_code, ds.pixel_array, final_seg, line_seg))
    except (IndexError, FileNotFoundError):
        print('ERROR loading', subj_code)

print(np.unique(labels))

XCF and DCM file reading:   0%|          | 0/343 [00:00<?, ?it/s]

1      ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
1.1    ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
1.2    ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
1.3    ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
2      ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
3      ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
3.2    ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
3.1    ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
4      ['cavum', 'cerebel', 'cisterna magna', 'midline', 'plec nucal', 'silvio'] (852, 1136) (852, 1136)
4.1    ['cavum', 'cerebel', 'cisterna magna', 'midline'

# Sylvian fissure fitting
The following cells focus on estimating the sylvian fissure as a mathematical function to estimate different properties (e.g. curvature, angles, lengths, etc.).

In [ ]:
def fit_polynomial(points_x, points_y, sub_code, degree=3, epochs=400000, tolerance = 1e-10):
    params = torch.rand(degree + 1, dtype=torch.float32, device=device, requires_grad=True)
    optimizer = optim.Adam([params], lr=0.01)
    ss_res = np.inf
    best_e = 0
    
    train_loop = tqdm(
        range(epochs), leave=False,
        desc=f"Sylvian fitting {sub_code}",
    )

    ss_tot = np.sum((points_y - points_y.mean()) ** 2)
    
    for e in train_loop:
        optimizer.zero_grad()
        
        pred_y = polynomial(params, torch.from_numpy(points_x).to(device=device))
        loss = F.mse_loss(pred_y, torch.from_numpy(points_y).to(device=device))
        loss.backward() # Backward pass
        optimizer.step() # Optimize

        loss_val = loss.detach().cpu().numpy()
        if ss_res > loss_val:
            improvement = ss_res - loss_val 
            best_e = e
            best_params = params.detach().clone() if not np.isinf(ss_res) else np.inf
            ss_res = loss_val
            r_2 = 1 - (ss_res / ss_tot)
            if improvement < tolerance:
                break

        train_loop.set_postfix(
            current_loss=f'{loss_val:.5f}',
            best_loss=f'{ss_res:.5f}',
            best_epoch=best_e
        )


    return best_params.detach().cpu().numpy(), ss_res, r_2


def polynomial(params, x):
    y = 0
    for idx, p in enumerate(params):
        y = y + p * x ** idx
    return y


fit_path = os.path.join(out_path, 'Lines')
Path(fit_path).mkdir(parents=True, exist_ok=True)

iters = 10
i = 0

red_cmap = ListedColormap([[0, 0, 0, 0], 'red'], name='foetal_us')
cyan_cmap = ListedColormap([[0, 0, 0, 0], 'skyblue'], name='foetal_us')

for sub, sub_data in global_dict.items():
    # if i == iters:
    #     break
    subj_code, sub_im, area_seg, line_seg = sub_data[0]
    midline = skeletonize(line_seg[0] > 0).astype(np.uint8)
    midline_y, midline_x = np.where(midline > 0)
    min_mpoint = np.argmin(midline_x)
    max_mpoint = np.argmax(midline_x)
    
    sylvian = skeletonize(line_seg[1] > 0).astype(np.uint8)
    sylvian_y, sylvian_x = np.where(sylvian > 0)
    min_spoint = np.argmin(sylvian_x)
    max_spoint = np.argmax(sylvian_x)
    vec_x = sylvian_x[max_spoint] - sylvian_x[min_spoint]
    vec_y = sylvian_y[max_spoint] - sylvian_y[min_spoint]
    length = np.sqrt(vec_x ** 2 + vec_y ** 2)
    cos_sub = vec_x / length
    sin_sub = vec_y / length
    angle = np.arccos(cos_sub)
    sylvian_x_c = sylvian_x - sylvian_x[min_spoint]
    sylvian_y_c = sylvian_y[min_spoint] - sylvian_y
    sylvian_x_norm = cos_sub * sylvian_x_c - sin_sub * sylvian_y_c
    sylvian_y_norm = sin_sub * sylvian_x_c + cos_sub * sylvian_y_c

    params, best_loss, r_2 = fit_polynomial(sylvian_x_norm, sylvian_y_norm, sub)
    min_x_norm = sylvian_x_norm[min_spoint]
    max_x_norm = sylvian_x_norm[max_spoint]
    x_range = np.linspace(min_x_norm, max_x_norm, 1000)
    pred_y = polynomial(params, x_range)

    new_sylvian_x = cos_sub * x_range + sin_sub * pred_y + sylvian_x[min_spoint]
    new_sylvian_y = sylvian_y[min_spoint] - (cos_sub * pred_y - sin_sub * x_range)
    print(sub, f'({sylvian_x[min_spoint]}, {sylvian_y[min_spoint]}) - ({sylvian_x[max_spoint]}, {sylvian_y[max_spoint]}) [{angle / np.pi * 180:.5f}] <best fit = {best_loss:.5f} | r^2 = {r_2:.5f}>')
    
    fig, axes = plt.subplots(1, 3, figsize=(60, 20))
    plt.subplot(1, 3, 1)
    plt.imshow(sub_im, cmap='gray')
    plt.imshow(line_seg[1].astype(np.uint8), cmap=red_cmap, vmin=0, alpha=0.50)
    axes[0].scatter(
        sylvian_x[min_spoint],
        sylvian_y[min_spoint],
        s=100, c='turquoise'
    )
    axes[0].scatter(
        sylvian_x[max_spoint],
        sylvian_y[max_spoint],
        s=100, c='green'
    )
    axes[0].quiver(
        sylvian_x[min_spoint], sylvian_y[min_spoint], vec_x, vec_y,
        scale_units='xy', scale=1,
        color='teal', angles='xy'
    )
    axes[0].set_title('Sylvian fissure (Subject {:>2})'.format(subj_code))
    axes[0].axis("off")
    plt.subplot(1, 3, 2)
    plt.imshow(sub_im, cmap='gray')
    plt.imshow(line_seg[0].astype(np.uint8), cmap=cyan_cmap, vmin=0, alpha=0.50)
    axes[1].scatter(
        midline_x[min_mpoint],
        midline_y[min_mpoint],
        s=100, c='turquoise'
    )
    axes[1].scatter(
        midline_x[max_mpoint],
        midline_y[max_mpoint],
        s=100, c='green'
    )
    axes[1].plot(
        [midline_x[min_mpoint], midline_x[max_mpoint]], 
        [midline_y[min_mpoint], midline_y[max_mpoint]],
        color='teal',
        linewidth=5
    )
    axes[1].set_title('Midline (Subject {:>2})'.format(subj_code))
    axes[1].axis("off")
    plt.subplot(1, 3, 3)
    plt.imshow(sub_im, cmap='gray')
    sorted_sylvian = np.argsort(sylvian_x)
    plt.imshow(line_seg[0].astype(np.uint8), cmap=cyan_cmap, vmin=0, alpha=0.50)
    plt.imshow(line_seg[1].astype(np.uint8), cmap=red_cmap, vmin=0, alpha=0.50)
    axes[2].plot(
        new_sylvian_x,
        new_sylvian_y,
        color='salmon',
        linewidth=2
    )
    axes[2].plot(
        midline_x,
        midline_y,
        color='cyan',
        linewidth=2
    )
    plt.tight_layout()
    plt.savefig(os.path.join(fit_path, f'{subj_code}_sylvian.png'))
    plt.close()
    i += 1

Sylvian fitting 1:   0%|          | 0/400000 [00:00<?, ?it/s]

1 (486, 619) - (560, 635) [12.20047] <best fit = 0.57119 | r^2 = 0.99984>


Sylvian fitting 2:   0%|          | 0/400000 [00:00<?, ?it/s]

2 (444, 665) - (575, 683) [7.82371] <best fit = 10.34681 | r^2 = 0.99939>


Sylvian fitting 3:   0%|          | 0/400000 [00:00<?, ?it/s]

...

Sylvian fitting 85:   0%|          | 0/400000 [00:00<?, ?it/s]

85 (554, 683) - (653, 669) [8.04906] <best fit = 0.80124 | r^2 = 0.99987>


Sylvian fitting 86:   0%|          | 0/400000 [00:00<?, ?it/s]

86 (535, 667) - (645, 649) [9.29331] <best fit = 5.38856 | r^2 = 0.99935>


Sylvian fitting 87:   0%|          | 0/400000 [00:00<?, ?it/s]

# Image segmentation
The following cells are a slight adaptation from older code to train a torchvision model to segment the annotated structures (masks, not points). This "newer" version also applies the trained models to the unsegmented images and visualizes the results.

In [ ]:
def run_segmentation_experiments(
    master_seed, network_name, display_name, experiment_name, network_f,
    data_set, weight_path, maps_path, classes=None, n_folds=5,  val_split=0.8, 
    epochs=10, patience=5,  n_seeds=1, n_inputs=3, n_classes=5, train_batch=2, test_batch=2,
    verbose=1
):
    # Make paths if they did not exist
    for d in [weight_path, maps_path]: Path(d).mkdir(parents=True, exist_ok=True)
    c = color_codes()

    subj_ids = list(data_set.keys())
    subs_x_fold = len(subj_ids) / n_folds

    # Choosing random runs.
    np.random.seed(master_seed)
    seeds = np.random.randint(0, 100000, n_seeds)
    
    # Main loop to run each independent random experiment.
    dsc_list = []
    class_dsc_list = []
    for test_n, seed in enumerate(seeds):
        if verbose > 1:
            print(
                '{:}[{:}] {:}Starting experiment {:}(seed {:05d})'
                '{:} [{:02d}/{:02d}] {:}for {:} segmentation{:}'.format(
                    c['clr'] + c['c'], strftime("%m/%d/%Y - %H:%M:%S"), c['g'],
                    c['nc'] + c['y'], seed, 
                    c['nc'] + c['c'], test_n + 1, len(seeds),
                    c['nc'] + c['g'], c['b'] + experiment_name + c['nc'] + c['g'], c['nc']
                )
            )

        np.random.seed(seed)
        torch.manual_seed(seed)

        # Dataset is a dictionary where the keys are subject, and for each key a list of tuples is given.
        rand_perm = np.random.permutation(subj_ids).tolist()

        seed_dsc_list = []
        seed_class_dsc_list = []
        
        for f_i in range(n_folds):            
            test_ini = int(round(f_i * subs_x_fold))
            test_end = int(round((f_i + 1) * subs_x_fold))
            test_ids = subj_ids[test_ini:test_end]
            train_val_ids = subj_ids[:test_ini] + subj_ids[test_end:]
            train_ids = train_val_ids[:int(len(train_val_ids) * val_split)]
            val_ids = train_val_ids[int(len(train_val_ids) * val_split):]
            training_images = [
                im for sub_id in train_ids for im, _, _ in data_set[sub_id]
            ]
            training_masks = [
                mask for sub_id in train_ids for _, mask, _ in data_set[sub_id]
            ]
            training_set = FetalDataset(training_images, training_masks)
            validation_images = [
                im for sub_id in val_ids for im, _, _ in data_set[sub_id]
            ]
            validation_masks = [
                mask for sub_id in val_ids for _, mask, _ in data_set[sub_id]
            ]
            validation_set = FetalDataset(validation_images, validation_masks)
            
            testing_images = [
                im for sub_id in test_ids for im, _, _ in data_set[sub_id]
            ]
            testing_masks = [
                mask for sub_id in test_ids for _, mask, _ in data_set[sub_id]
            ]
            testing_set = FetalDataset(testing_images, testing_masks)
    
            # The network will only be instantiated with the number of output classes
            # (2 in this notebok). Therefore, networks that need extra parameters (like ViT)
            # will need to be passed as a partial function.
            net = network_f(n_inputs=n_inputs, n_outputs=n_classes)
    
            # This is a leftover from legacy code. If init is set to True (the default option),
            # a first validation epoch will be run to determine the loss before training.
            net.init = False
    
            # The number of parameters is only captured for debugging and printing.
            n_param = sum(
                p.numel() for p in net.parameters() if p.requires_grad
            )

            if verbose > 1:
                print(
                    '   {:}Fold {:}{:02d}/{:02d} {:}({:} - {:}[{:,} parameters]{:}){:}'.format(
                        c['g'], c['nc'] + c['c'], f_i + 1, n_folds,
                        c['y'], c['b'] + display_name,
                        c['nc'], n_param, c['y'], c['nc']
                    )
                )
            print(
                '   |_ Training', len(training_set), '[{:d} subjects]'.format(len(train_ids))
            )
            print(
                '   |_ Validation', len(validation_set), '[{:d} subjects]'.format(len(val_ids))
            )
            print(
                '   |_ Testing', len(testing_set), '[{:d} subjects]'.format(len(test_ids)),
            )
    
            training_loader = DataLoader(
                training_set, train_batch, True
            )
            validation_loader = DataLoader(
                validation_set, test_batch
            )
            model_path = os.path.join(
                weight_path, '{:}-balanced_s{:05d}_f{:01d}.pt'.format(network_name, seed, f_i)
            )
    
            # For efficiency, we only run the code once. If the weights are
            # stored on disk, we do not need to train again.
            try:
                net.load_model(model_path)
            except IOError:
                net.train()
                print(''.join([' '] * 200), end='\r')
                net.fit(training_loader, validation_loader, epochs=epochs, patience=patience)
                net.save_model(model_path)
    
            if verbose > 2:
                print(''.join([' '] * 200), end='\r')
                print(
                    '   {:}Testing <{:02d} samples>{:}'.format(
                        c['g'], len(testing_set), c['nc']
                    )
                )
    
            # Metric evaluation.
            net.eval()
            with torch.no_grad():
                mosaic_dsc = []
                mosaic_class_dsc = []
                # Intermediate buffers for class metrics.
                #for input_mosaic, mask_i in zip(testing_mosaics, testing_masks):
                im_i = 0
                for tst_i, (input_mosaic, mask_i) in enumerate(testing_set):
                    pred_map = net.inference(
                        np.expand_dims(
                            input_mosaic.astype(np.float32),
                            axis=0
                        )
                    )[0]
    
                    pred_y = np.argmax(pred_map, axis=0).astype(np.uint8)
                    y = mask_i.astype(np.uint8)
                    intersection = np.stack([
                        2 * np.sum(np.logical_and(pred_y == lab, y == lab))
                        for lab in range(n_classes)
                    ])
                    card_pred_y = np.stack([
                        np.sum(pred_y == lab) for lab in range(n_classes)
                    ])
                    card_y = np.stack([
                        np.sum(y == lab) for lab in range(n_classes)
                    ])
                    dsc_k = intersection / (card_pred_y + card_y)
                    dsc = np.nanmean(dsc_k)
                    dsc_std = np.nanstd(dsc_k)
                    mosaic_dsc.append(dsc)
                    mosaic_class_dsc.append(dsc_k.tolist())
    
                    seg_map = (np.argmax(pred_map, axis=0) * 55).astype(np.uint8)
                    
                    dsc = np.nanmean(mosaic_dsc, axis=0)
                    class_dsc = np.nanmean(mosaic_class_dsc, axis=0)
                    class_std = np.nanstd(mosaic_class_dsc, axis=0)
                    if tst_i % 20 == 0:
                        # Plotting the segmentations
                        fig, axes = plt.subplots(1, 2, figsize=(20, 40))
                        plt.subplot(1, 2, 1)
                        plt.imshow(input_mosaic[0, ...], cmap='gray')
                        plt.imshow(mask_i, cmap='tab10', vmin=0, alpha=0.50)
                        axes[0].set_title(
                            'Ground truth (ID: {:02d} / seed {:} / fold {:02d})'.format(
                                tst_i, seed, f_i
                            )
                        )
                        axes[0].axis("off")
                        plt.subplot(1, 2, 2)
                        plt.imshow(input_mosaic[0, ...], cmap='gray')
                        plt.imshow(np.argmax(pred_map, axis=0), cmap='tab10', vmin=0, alpha=0.50)
                        axes[1].set_title(
                            'Test prediction (ID: {:02d} / seed {:} / fold {:02d})'.format(
                                tst_i, seed, f_i
                            )
                        )
                        axes[1].axis("off")
                        
        
            if verbose > 2:
                print(''.join([' '] * 200), end='\r')
                print(
                    '   {:}Fold {:}{:02d}/{:02d} {:}DSC{:} (seed {:05d}){:} [{:02d}/{:02d}] {:}'
                    '{:5.3f} ± {:5.3f}{:}'.format(
                        c['g'], c['nc'] + c['c'], f_i + 1, n_folds, c['g'],
                        c['nc'] + c['y'], seed, c['nc'] + c['c'], test_n + 1, len(seeds),
                        c['nc'] + c['b'], dsc, dsc_std, c['nc']
                    )
                )
    
                class_dsc_s = ', '.join([
                '{:} {:5.3f} ± {:5.3f}'.format(k, dsc_k, std_k)
                    for k, dsc_k, std_k in zip(classes, class_dsc, class_std)
                ])
                print(
                    '   {:}Fold {:}{:02d}/{:02d} {:}Class DSC{:} (seed {:05d}){:} [{:02d}/{:02d}] {:}'.format(
                        c['g'], c['nc'] + c['c'], f_i + 1, n_folds, c['g'],
                        c['nc'] + c['y'], seed, c['nc'] + c['c'], test_n + 1, len(seeds),
                        c['nc'] + c['b'] + class_dsc_s + c['nc']
                    )
                )
            elif verbose > 1:
                print(''.join([' '] * 200), end='\r')
                print(
                    '{:}Seed {:05d} {:} [{:,} parameters] '
                    '{:}[{:02d}/{:02d}] {:} {:5.3f}{:}'.format(
                        c['y'], seed, c['b'] + display_name + c['nc'], n_param,
                        c['c'], test_n + 1, len(seeds),
                        c['nc'] + c['g'] + c['b'] + experiment_name + c['nc'] + c['b'],
                        dsc, c['nc']
                    )
                )
            elif verbose > 0:
                print(''.join([' '] * 200), end='\r')
                print(
                    '{:}Seed {:05d} {:} [{:,} parameters] '
                    '{:}[{:02d}/{:02d}] {:} {:5.3f}{:}'.format(
                        c['y'], seed, c['b'] + display_name + c['nc'], n_param,
                        c['c'], test_n + 1, len(seeds),
                        c['nc'] + c['g'] + c['b'] + experiment_name + c['nc'] + c['b'],
                        dsc, c['nc']
                    ), end='\r'
                )
            net = None
            seed_dsc_list.append(mosaic_dsc)
            seed_class_dsc_list.append(mosaic_class_dsc)

        if verbose > 0:
            seed_mean_dsc = np.nanmean(np.concatenate(seed_dsc_list))
            seed_std_dsc = np.nanstd(np.concatenate(seed_dsc_list))
            seed_class_mean_dsc = np.nanmean(
                np.concatenate(seed_class_dsc_list, axis=0), axis=0
            )
            seed_class_std_dsc = np.nanstd(
                np.concatenate(seed_class_dsc_list, axis=0), axis=0
            )
            print(''.join([' '] * 200), end='\r')
            print(
                '   {:}Seed {:05d} - mean DSC{:} {:5.3f} ± {:5.3f}{:}'.format(
                    c['y'] + c['b'] + display_name + c['nc'] + c['g'],
                    c['nc'] + c['b'], seed_mean_dsc, seed_std_dsc, c['nc']
                )
            )
            class_dsc_s = ', '.join([
                '{:} {:5.3f} ± {:5.3f}'.format(k, dsc_k, std_k)
                for k, dsc_k, std_k in zip(
                    classes, seed_class_mean_dsc, seed_class_std_dsc
                )
            ])
            print(
                '   {:}Seed {:05d} - mean class DSC {:}'.format(
                    c['y'] + c['b'] + display_name + c['nc'] + c['g'],
                    c['nc'] + c['b'] + class_dsc_s + c['nc']
                )
            )
        dsc_list.append(seed_mean_dsc)
        class_dsc_list.append(seed_class_mean_dsc)

    # Metrics for all the runs.
    if verbose > 0:
        print(''.join([' '] * 200), end='\r')
        print(
            '{:}[{:}] {:} Overall mean DSC{:} {:5.3f}{:}'.format(
                c['clr'] + c['c'], strftime("%m/%d/%Y - %H:%M:%S"),
                c['nc'] + c['y'] + c['b'] + display_name + c['nc'] + c['g'],
                c['nc'] + c['b'], np.nanmean(dsc_list), c['nc']
            )
        )
        class_dsc_s = ', '.join([
            '{:} {:5.3f}'.format(k, dsc_k)
            for k, dsc_k in zip(
                classes, np.nanmean(class_dsc_list, axis=(0, 1))
            )
        ])
        print(
            '{:}[{:}] {:} Mean class DSC {:}'.format(
                c['clr'] + c['c'], strftime("%m/%d/%Y - %H:%M:%S"),
                c['nc'] + c['y'] + c['b'] + display_name + c['nc'] + c['g'],
                c['nc'] + c['b'] + class_dsc_s + c['nc']
            )
        )

    return dsc_list, class_dsc_list

In [ ]:
master_seed = 42

np.random.seed(master_seed)
torch.manual_seed(master_seed)

train_batch = 1
test_batch = 1
epochs = 10
patience = 5
n_seeds = 5
name = 'fetal us'
classes = ['background'] + layer_names[:-2]

# The experiments are run next. We capture some warnings related to
# image loading to clean the debugging console.
# FCN ResNet50
fcn50_dsc, fcn50_k_dsc = run_segmentation_experiments(
    master_seed, 'fcn-resnet50', 'FCN ResNet50', name, partial(FCN_ResNet50, lr=1e-4, pretrained=True),
    global_dict, os.path.join(out_path, 'Weights'), os.path.join(out_path, 'Predictions'),
    classes, n_inputs=3, n_classes=len(classes), epochs=epochs, patience=patience,
    train_batch=train_batch, test_batch=test_batch, n_seeds=n_seeds, verbose=3
)